In [4]:
!rm -rf __pycache__/

In [5]:
# Clean up old installs
!pip uninstall -y transformers tokenizers huggingface-hub
!rm -rf /usr/local/lib/python3.11/dist-packages/transformers
!rm -rf /usr/local/lib/python3.11/dist-packages/tokenizers
!rm -rf /usr/local/lib/python3.11/dist-packages/huggingface_hub

# Reinstall stable, tested version
!pip install transformers==4.39.3


Found existing installation: transformers 4.53.1
Uninstalling transformers-4.53.1:
  Successfully uninstalled transformers-4.53.1
Found existing installation: tokenizers 0.21.2
Uninstalling tokenizers-0.21.2:
  Successfully uninstalled tokenizers-0.21.2
Found existing installation: huggingface-hub 0.33.2
Uninstalling huggingface-hub-0.33.2:
  Successfully uninstalled huggingface-hub-0.33.2
  Using cached transformers-4.39.3-py3-none-any.whl.metadata (134 kB)
  Using cached huggingface_hub-0.33.2-py3-none-any.whl.metadata (14 kB)
  Using cached tokenizers-0.15.2-cp311-cp311-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (6.7 kB)
Using cached transformers-4.39.3-py3-none-any.whl (8.8 MB)
Using cached huggingface_hub-0.33.2-py3-none-any.whl (515 kB)
Using cached tokenizers-0.15.2-cp311-cp311-manylinux_2_17_x86_64.manylinux2014_x86_64.whl (3.6 MB)
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source

In [1]:
import transformers
print(transformers.__version__)

4.35.2


/usr/local/lib/python3.11/dist-packages/transformers/utils/generic.py:441: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  _torch_pytree._register_pytree_node(


In [1]:
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer, DataCollatorForSeq2Seq

model_name = 'google/flan-t5-base'
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)
data_collator = DataCollatorForSeq2Seq(tokenizer, model=model)


/usr/local/lib/python3.11/dist-packages/huggingface_hub/file_download.py:943: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/huggingface_hub/file_download.py:943: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possibl

In [2]:
def preprocess(example):
    input_texts = [f"Question: {prompt} Answer:" for prompt in example["prompt"]]
    target_texts = example["response"]

    model_inputs = tokenizer(
        input_texts,
        max_length=256,
        padding="max_length",
        truncation=True,
    )

    with tokenizer.as_target_tokenizer():
        labels = tokenizer(
            target_texts,
            max_length=128,
            padding="max_length",
            truncation=True,
        )

    model_inputs["labels"] = labels["input_ids"]
    return model_inputs


In [3]:
import json
from datasets import Dataset

# Define the path to your JSONL file
file_path = "/content/bit_dialogues_high_quality_expanded.jsonl"

# Read the data from the JSONL file
data = []
with open(file_path, 'r') as f:
    for line in f:
        data.append(json.loads(line))

# Create a Dataset object from the list of dictionaries
raw_dataset = Dataset.from_list(data)

# Split the dataset
split_dataset = raw_dataset.train_test_split(test_size=0.1)

train_dataset = split_dataset["train"]
eval_dataset = split_dataset["test"]

print("Dataset loaded and split successfully!")
print("Train dataset size:", len(train_dataset))
print("Eval dataset size:", len(eval_dataset))

Dataset loaded and split successfully!
Train dataset size: 39
Eval dataset size: 5


In [4]:
# Preprocess both train and eval datasets
tokenized_train = train_dataset.map(preprocess, batched=True, remove_columns=["prompt", "response"])
tokenized_eval = eval_dataset.map(preprocess, batched=True, remove_columns=["prompt", "response"])


Map:   0%|          | 0/39 [00:00<?, ? examples/s]

/usr/local/lib/python3.11/dist-packages/transformers/tokenization_utils_base.py:3935: UserWarning: `as_target_tokenizer` is deprecated and will be removed in v5 of Transformers. You can tokenize your labels by using the argument `text_target` of the regular `__call__` method (either in the same call as your input texts if you use the same keyword arguments, or in a separate call.
  warnings.warn(


Map:   0%|          | 0/5 [00:00<?, ? examples/s]

In [5]:
!pip install evaluate
!pip install rouge_score


In [6]:
import evaluate

rouge = evaluate.load("rouge")

def compute_metrics(eval_pred):
    predictions, labels = eval_pred

    decoded_preds = tokenizer.batch_decode(predictions, skip_special_tokens=True)
    decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)

    decoded_preds = [pred.strip() for pred in decoded_preds]
    decoded_labels = [label.strip() for label in decoded_labels]

    result = rouge.compute(predictions=decoded_preds, references=decoded_labels)

    return {
        "rouge1": result["rouge1"],
        "rouge2": result["rouge2"],
        "rougeL": result["rougeL"]
    }


In [7]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="./flan_t5_bit_model",
    evaluation_strategy="epoch",
    save_strategy="epoch",
    logging_strategy="steps",
    logging_steps=5,
    per_device_train_batch_size=4,
    num_train_epochs=10,
    learning_rate=3e-5,
    warmup_steps=100,
    weight_decay=0.01,
    save_total_limit=1,
    load_best_model_at_end=True,
    metric_for_best_model="rougeL",
    greater_is_better=True,
    # fp16=True
)


In [8]:
from transformers import EarlyStoppingCallback

In [9]:
!pip install  peft

In [10]:
import transformers
print(transformers.__version__)

4.39.3


In [11]:
from transformers import Trainer

RuntimeError: Failed to import transformers.trainer because of the following error (look up to see its traceback):
cannot import name 'EncoderDecoderCache' from 'transformers' (/usr/local/lib/python3.11/dist-packages/transformers/__init__.py)

In [2]:


trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_eval,
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)]
)

NameError: name 'model' is not defined

In [4]:
from transformers import Trainer

# Make sure to run the cells that define `model`, `tokenizer`, and `data_collator` first
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_eval,
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)]  # ✅ NEW
)


trainer.train()

NameError: name 'model' is not defined

In [24]:
def get_response(prompt):
    inputs = tokenizer(f"Question: {prompt} Answer:", return_tensors="pt")
    inputs = {k: v.to(model.device) for k, v in inputs.items()}
    generation_config = GenerationConfig(
        max_new_tokens=100,
        num_beams=4,
        no_repeat_ngram_size=2,
        early_stopping=True
    )
    output = tokenizer.decode(
        model.generate(
            inputs["input_ids"],
            generation_config=generation_config
        )[0],
        skip_special_tokens=True
    )

    if "I donape don't know" in output or "I cannot answer" in output or "computer program" in output:
        return "I'm sorry, I couldn't find the information you're looking for. Please try a different question or contact us directly."

    return output

# Example usage
sentence = "User: Where is BIT located? ?Assistant: "
response = get_response(sentence.split(": ")[1].strip())
print(response)

In the United States


In [ ]:
def get_response(prompt):
    inputs = tokenizer(f"Question: {prompt} Answer:", return_tensors="pt")
    inputs = {k: v.to(model.device) for k, v in inputs.items()}
    generation_config = GenerationConfig(
        max_new_tokens=100,
        num_beams=4,
        no_repeat_ngram_size=2,
        early_stopping=True
    )
    output = tokenizer.decode(
        model.generate(
            inputs["input_ids"],
            generation_config=generation_config
        )[0],
        skip_special_tokens=True
    )

    if "I don't know" in output or "I cannot answer" in output or "computer program" in output:
        return "I'm sorry, I couldn't find the information you're looking for. Please try a different question or contact us directly."

    return output

# Example usage
sentence = "User: Where is BIT located? ?Assistant: "
response = get_response(sentence.split(": ")[1].strip())
print(response)

In [ ]:
model.save_pretrained("flan_t5_bit_model")
tokenizer.save_pretrained("flan_t5_bit_model")

!zip -r flan_t5_bit_model.zip flan_t5_bit_model/
from google.colab import files
files.download("flan_t5_bit_model.zip")